# Example: Rebalancing Engine and Scorecard

In this example, we wire the **Cobb-Douglas utility allocator** into a full rebalancing engine with trigger rules (drawdown limits, turnover caps), produce a scorecard comparing the engine to baseline strategies, and run a sensitivity analysis over the key parameters $\lambda$ (risk-aversion) and $\epsilon$ (minimum share floor).

> **By the end of this example, you will be able to:**
> * Run the Cobb-Douglas rebalancing engine with production-style trigger rules
> * Produce a scorecard comparing the AI engine to buy-and-hold and risk-free benchmarks
> * Understand how lambda gain and epsilon affect portfolio performance via sensitivity analysis

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

Load the data from Example 1 — market data, engine context, and benchmark wealth series.

In [ ]:
let
    # Load data from Example 1 -
    data = load(joinpath(_PATH_TO_DATA, "engine-run-data.jld2"));

    global price_matrix = data["price_matrix"];
    global lambda_series = data["lambda_series"];
    global market_prices = data["market_prices"];
    global gm_ema = data["gm_ema"];
    global tickers = data["tickers"];
    global sim_params = data["sim_params"];
    global context = data["context"];
    global buyhold_wealth = data["buyhold_wealth"];
    global riskfree_wealth = data["riskfree_wealth"];

    # Constants -
    global Δt = 1.0 / 252.0;
    global N_short = 21;
    global N_long = 63;
    global offset = N_short + N_long;
    global n_trading_days = 252;
    global B₀ = context.B;
    global K = length(tickers);

    println("Loaded engine data: $(K) tickers, $(n_trading_days) trading days")
end

___
## Task 1: Cobb-Douglas Rebalancing Engine with Trigger Rules
We run the Cobb-Douglas utility allocator inside the full rebalancing engine with realistic trigger rules: a 15% drawdown limit (circuit breaker) and a 50% turnover cap (trading cost control). We compare three drawdown thresholds to see how the safety net affects performance.

> **What should you see?** Tighter drawdown limits (10%) cause the engine to de-risk to cash earlier and more often — protecting capital but potentially missing recoveries. Looser limits (25%) allow more volatility. The engine uses Cobb-Douglas utility to decide _how_ to allocate; the trigger rules decide _whether_ to allocate.

In [ ]:
let
    drawdown_limits = [0.10, 0.15, 0.25];
    colors = [:red :orange :green];
    labels = ["DD ≤ 10%", "DD ≤ 15%", "DD ≤ 25%"];

    global dd_wealth_curves = Dict{Float64, Array{Float64,1}}();

    p = plot(size=(800, 450), title="Cobb-Douglas Engine: Drawdown-Triggered De-Risking",
        xlabel="Trading Day (after warmup)", ylabel="Wealth (\$)", legend=:topleft)

    for (j, dd_limit) in enumerate(drawdown_limits)

        rules = build(MyTriggerRules, (
            max_drawdown = dd_limit,
            max_turnover = 0.50,
            rebalance_schedule = ones(Int, n_trading_days)
        ));

        results = run_rebalancing_engine(context, rules, lambda_series;
            offset=offset, allocator=:cobb_douglas);
        wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);
        dd_wealth_curves[dd_limit] = wealth;

        plot!(p, 1:length(wealth), wealth, label=labels[j], linewidth=2, color=colors[j])
    end

    # add buy-and-hold reference -
    plot!(p, 1:length(buyhold_wealth), buyhold_wealth, label="Buy-and-Hold",
        linewidth=1.5, color=:gray50, linestyle=:dash)
    p
end

___
## Task 2: Scorecard — Engine vs. Baselines
We produce a comprehensive scorecard comparing three strategies: the AI Cobb-Douglas engine (DD $\leq$ 15%, $\tau \leq$ 50%), equal-weight buy-and-hold, and risk-free. The scorecard tracks return, volatility, Sharpe ratio, and maximum drawdown.

> **What should you see?** The engine should show better risk-adjusted performance (lower drawdown, potentially better Sharpe) than static allocations, at the cost of higher trading activity. The scorecard quantifies exactly how much adaptability costs and what it buys.

In [ ]:
let
    # Run the moderate engine configuration -
    rules = build(MyTriggerRules, (
        max_drawdown = 0.15, max_turnover = 0.50,
        rebalance_schedule = ones(Int, n_trading_days)
    ));
    results = run_rebalancing_engine(context, rules, lambda_series;
        offset=offset, allocator=:cobb_douglas);
    global engine_wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);

    # Helper: compute metrics from a wealth series -
    function scorecard_metrics(wealth::Array{Float64,1}, label::String)
        returns = diff(wealth) ./ wealth[1:end-1];
        total_return = (wealth[end] / wealth[1] - 1.0) * 100;
        vol = std(returns) * sqrt(252) * 100;
        sharpe = vol > 0 ? total_return / vol : 0.0;
        peak = accumulate(max, wealth);
        dd = maximum((peak .- wealth) ./ peak) * 100;
        return (label, round(total_return, digits=2), round(vol, digits=2),
            round(sharpe, digits=2), round(dd, digits=2))
    end

    # Compute metrics for each strategy -
    eng = scorecard_metrics(engine_wealth, "Cobb-Douglas Engine (DD≤15%, τ≤50%)");
    bh = scorecard_metrics(buyhold_wealth, "Buy-and-Hold (equal-weight)");
    rf = scorecard_metrics(collect(riskfree_wealth), "Risk-Free (4.3%)");

    # Build scorecard table -
    scorecard = DataFrame(
        "Strategy" => [eng[1], bh[1], rf[1]],
        "Return (%)" => [eng[2], bh[2], rf[2]],
        "Volatility (%)" => [eng[3], bh[3], rf[3]],
        "Sharpe Ratio" => [eng[4], bh[4], rf[4]],
        "Max Drawdown (%)" => [eng[5], bh[5], rf[5]]
    );

    println("═"^70)
    println("  SESSION 2 SCORECARD: Cobb-Douglas Engine vs. Baselines")
    println("═"^70)
    pretty_table(scorecard, tf=tf_markdown)

    # Save for Session 3 -
    save(joinpath(_PATH_TO_DATA, "session2-scorecard.jld2"),
        Dict("scorecard" => scorecard, "engine_wealth" => engine_wealth,
             "buyhold_wealth" => buyhold_wealth));
end

**Visualize:** Wealth curves — all three strategies.

In [ ]:
let
    days = 1:length(engine_wealth);

    plot(days, engine_wealth, label="Cobb-Douglas Engine", linewidth=2.5, color=:steelblue)
    plot!(days, buyhold_wealth, label="Buy-and-Hold", linewidth=2, color=:coral, linestyle=:dash)
    plot!(days, collect(riskfree_wealth), label="Risk-Free (4.3%)", linewidth=1.5, color=:gray50, linestyle=:dot)
    xlabel!("Trading Day (after warmup)")
    ylabel!("Wealth (\$)")
    title!("Session 2: Cobb-Douglas Rebalancing Engine vs. Baselines")
    plot!(size=(800, 450), legend=:topleft)
end

___
## Task 3: Sensitivity Analysis — Lambda Gain and Epsilon
We sweep two key parameters to understand how they affect portfolio performance:

1. **Lambda gain $G$** — controls how aggressively the sentiment signal modulates risk. Higher $G$ means the engine reacts more strongly to EMA crossover.
2. **Epsilon $\epsilon$** — the minimum share floor for non-preferred assets. Higher $\epsilon$ means more capital is locked in non-preferred positions (more diversified but less responsive).

> **What should you see?** A heatmap showing how Sharpe ratio (or final wealth) varies across the $(G, \epsilon)$ parameter space. There should be a "sweet spot" — too low $G$ makes the engine unresponsive; too high $G$ makes it over-reactive. Too low $\epsilon$ concentrates risk; too high $\epsilon$ wastes budget on non-preferred assets.

In [ ]:
let
    # Parameter grids -
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];

    # Storage for Sharpe ratios -
    global sharpe_grid = zeros(length(G_values), length(ε_values));
    global wealth_grid = zeros(length(G_values), length(ε_values));

    for (i, G) in enumerate(G_values)
        for (j, ε) in enumerate(ε_values)

            # Recompute lambda with this gain -
            ema_s = compute_ema(market_prices; window=21);
            ema_l = compute_ema(market_prices; window=63);
            λ_new = compute_lambda(ema_s, ema_l; G=G);
            λ_new[1:offset] .= 0.0;

            # Build context with this epsilon -
            ctx = build(MyRebalancingContextModel, (
                B = B₀, tickers = tickers, marketdata = price_matrix,
                marketfactor = gm_ema, sim_parameters = sim_params,
                lambda = 0.0, Δt = Δt, epsilon = ε
            ));

            rules = build(MyTriggerRules, (
                max_drawdown = 0.15, max_turnover = 0.50,
                rebalance_schedule = ones(Int, n_trading_days)
            ));

            results = run_rebalancing_engine(ctx, rules, λ_new;
                offset=offset, allocator=:cobb_douglas);
            wealth = compute_wealth_series(results, price_matrix, tickers; offset=offset);

            # Compute Sharpe -
            returns = diff(wealth) ./ wealth[1:end-1];
            total_ret = (wealth[end] / wealth[1] - 1.0);
            vol = std(returns) * sqrt(252);
            sharpe_grid[i, j] = vol > 0 ? total_ret / vol : 0.0;
            wealth_grid[i, j] = wealth[end];
        end
    end

    println("Sensitivity analysis complete: $(length(G_values)) × $(length(ε_values)) grid")
end

**Visualize:** Heatmap of Sharpe ratio across the $(G, \epsilon)$ parameter space.

In [ ]:
let
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];

    heatmap(string.(ε_values), string.(G_values), sharpe_grid,
        xlabel="ε (min share floor)", ylabel="G (lambda gain)",
        title="Sharpe Ratio: Sensitivity to G and ε",
        color=:viridis, size=(700, 500),
        clim=(minimum(sharpe_grid), maximum(sharpe_grid)))
end

**Visualize:** Heatmap of final wealth across the same parameter space.

In [ ]:
let
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];

    heatmap(string.(ε_values), string.(G_values), wealth_grid,
        xlabel="ε (min share floor)", ylabel="G (lambda gain)",
        title="Final Wealth (\$): Sensitivity to G and ε",
        color=:viridis, size=(700, 500))
end

**Summary table:** Best and worst parameter combinations.

In [ ]:
let
    G_values = [2.0, 5.0, 10.0, 15.0, 20.0];
    ε_values = [0.01, 0.05, 0.1, 0.5, 1.0];

    # Find best and worst -
    best_idx = argmax(sharpe_grid);
    worst_idx = argmin(sharpe_grid);

    println("Best Sharpe:  G=$(G_values[best_idx[1]]), ε=$(ε_values[best_idx[2]]) → Sharpe=$(round(sharpe_grid[best_idx], digits=3)), Wealth=\$$(round(wealth_grid[best_idx], digits=2))")
    println("Worst Sharpe: G=$(G_values[worst_idx[1]]), ε=$(ε_values[worst_idx[2]]) → Sharpe=$(round(sharpe_grid[worst_idx], digits=3)), Wealth=\$$(round(wealth_grid[worst_idx], digits=2))")
end

___
## Summary
In this example, we ran the Cobb-Douglas rebalancing engine with production-style trigger rules and produced a comprehensive analysis:

- **Drawdown limits** act as circuit breakers — tighter limits protect capital but miss recoveries
- **The scorecard** quantifies the trade-off: the Cobb-Douglas engine adapts to market conditions at the cost of turnover
- **Sensitivity analysis** reveals the parameter sweet spot: how aggressively should the engine respond to sentiment ($G$) and how much budget should be reserved for non-preferred assets ($\epsilon$)

The engine results and scorecard are saved for Session 3, where we'll stress-test across HMM-generated regime-switching paths and introduce a **bandit challenger** to compete with the utility-based allocator.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.